# 🔄 Módulo 09 — Orchestration: Sequential

> **Objetivo:** Construir un pipeline de agentes donde cada uno recibe el output completo del anterior.

📚 **Doc oficial:** [Sequential Orchestration (Python)](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/sequential?pivots=programming-language-python)

## ¿Qué es Sequential Orchestration?

Los agentes se organizan en **pipeline** — cada uno procesa la tarea por turno y pasa su output al siguiente.

```
Usuario  ──→  [Writer]  ──→  [Reviewer]  ──→  [Editor]  ──→  Output final
```

Ideal para:
- Generación + revisión de contenido
- Pipelines multi-etapa de razonamiento
- Traducción / refinamiento progresivo

## API clave

```python
from agent_framework.orchestrations import SequentialBuilder

workflow = SequentialBuilder(participants=[writer, reviewer]).build()
events = await workflow.run("Escribe un slogan...")
outputs = events.get_outputs()       # list[AgentResponse]
```

> Por defecto cada agente ve la **conversación completa**. Usa `chain_only_agent_responses=True`
> si quieres que cada agente vea sólo la respuesta del anterior (útil para traducción).


## ⚙️ Setup

Carga el ChatClient compartido.

In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados.")


## 1️⃣ Pipeline básico: Writer → Reviewer

`SequentialBuilder` ejecuta los participantes en orden, propagando el contexto de la conversación.
El output final es un `AgentResponse` del último agente.


In [ ]:
from agent_framework import AgentResponse, Message
from agent_framework.orchestrations import SequentialBuilder

client = create_chat_client()

writer = Agent(
    client=client,
    name="writer",
    instructions=(
        "Eres un redactor publicitario conciso. Proporciona una sola oración de marketing impactante "
        "based on the prompt."
    ),
)

reviewer = Agent(
    client=client,
    name="reviewer",
    instructions=(
        "Eres un revisor analítico. Da retroalimentación breve sobre el mensaje anterior del asistente."
    ),
)

workflow = SequentialBuilder(participants=[writer, reviewer]).build()

events = await workflow.run("Escribe un eslogan para una bicicleta eléctrica económica.")
outputs = events.get_outputs()

print("===== Final Conversation =====\n")
if outputs:
    # `outputs[0]` es una lista de Message (la conversación completa)
    conversation = outputs[0] if isinstance(outputs[0], list) else [outputs[0]]
    for msg in conversation:
        name = getattr(msg, "author_name", None) or getattr(msg, "role", "assistant")
        print(f"[{name}]\n{msg.text}\n")


## 2️⃣ Streaming + intermediate outputs

Cuando quieres ver lo que cada agente produce **en tiempo real**, usa `stream=True`.
Por defecto sólo el último agente emite `"output"`. Para ver los intermedios, declara
`intermediate_output_from=[...]` y procesa eventos `"intermediate"`.


In [ ]:
from agent_framework import AgentResponseUpdate

editor = Agent(
    client=client,
    name="editor",
    instructions="Pule el eslogan basándote en la retroalimentación del revisor. Devuelve SOLO el eslogan final.",
)

# `intermediate_outputs=True` permite que cada participante emita su output como
# evento "intermediate" (en lugar de sólo el último como "output").
workflow = SequentialBuilder(
    participants=[writer, reviewer, editor],
    intermediate_outputs=True,
).build()

last_author = None
async for event in workflow.run(
    "Escribe un eslogan para un scooter eléctrico ecológico.",
    stream=True,
):
    if event.type in ("output", "intermediate") and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            label = "FINAL" if event.type == "output" else "intermediate"
            print(f"[{label}] {author}: {update.text}", end="", flush=True)
            last_author = author
        else:
            print(update.text, end="", flush=True)
print()


## 3️⃣ `chain_only_agent_responses=True` — pipeline de traducción

Por defecto cada agente ve la conversación completa. Con `chain_only_agent_responses=True`,
cada agente sólo recibe el **mensaje del anterior** — útil para pipelines de refinamiento puro
(traducción, resumen progresivo, etc.) donde no quieres que el agente se distraiga con el prompt original.


In [ ]:
spanish_translator = Agent(
    client=client,
    name="spanish_translator",
    instructions="Traduce el texto de entrada al inglés. Responde SOLO con la traducción.",
)

french_translator = Agent(
    client=client,
    name="french_translator",
    instructions="Traduce el texto de entrada al francés. Responde SOLO con la traducción.",
)

workflow = SequentialBuilder(participants=[spanish_translator, french_translator]).build()

events = await workflow.run("El futuro pertenece a quienes se preparan para él hoy.")
outputs = events.get_outputs()
if outputs:
    for msg in (outputs[0] if isinstance(outputs[0], list) else [outputs[0]]):
        print(f"[{msg.author_name}]: {msg.text}")


## 🎯 Resumen

| Capacidad | API |
|-----------|-----|
| Pipeline básico | `SequentialBuilder(participants=[a, b]).build()` |
| Streaming | `async for event in workflow.run(prompt, stream=True):` |
| Outputs intermedios | `intermediate_output_from=[agent_a, agent_b]` |
| Cada agente ve sólo el anterior | `chain_only_agent_responses=True` |
| Pausa para revisión humana | `.with_request_info(agents=[...])` |

**Siguiente módulo →** [Módulo 10 — Orchestration Concurrent](./10_orchestration_concurrent.ipynb)
